In [1]:
import json
from pathlib import Path
import torch
from torch import nn
from datasets import Dataset
from datasets import ClassLabel
from setfit import SetFitModel, Trainer, TrainingArguments
import numpy as np
from src.classifier.intent_classifier import IntentNN
from src.config import TRAINING_DATA_PATH

/home/asunaron/CLAWS/CORVUS_PythonServer/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

try:
    with open(TRAINING_DATA_PATH, "r") as f:
        data = json.load(f)
except FileNotFoundError:
    print("File not found!")
except json.JSONDecodeError:
    print("Invalid JSON format!")

In [68]:
texts = []
labels = []
label_to_idx = {}

for idx, intent in enumerate(data['intents']):
    label_to_idx[intent['label']] = idx
    for example in intent['examples']:
        texts.append(example)
        labels.append(idx)

In [69]:
dataset = Dataset.from_dict({"text": texts, "label": labels})

label_names = dataset.unique("label")
dataset = dataset.cast_column("label", ClassLabel(names=sorted(label_names)))

split = dataset.train_test_split(test_size=0.2, stratify_by_column="label")
train_dataset = split["train"]
eval_dataset = split["test"]

Casting the dataset: 100%|██████████| 1246/1246 [00:00<00:00, 305334.35 examples/s]

In [70]:
SF_model = SetFitModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

args = TrainingArguments(
    num_iterations=5,
    num_epochs=1,
)

trainer = Trainer(
    model=SF_model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

trainer.train()
results = trainer.evaluate()
print(results)

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Map: 100%|██████████| 996/996 [00:00<00:00, 25645.99 examples/s]
***** Running training *****
  Num unique pairs = 9960
  Batch size = 16
  Num epochs = 1


Step,Training Loss
1,0.099600
50,0.110700
100,0.077400
150,0.059000
200,0.051700
250,0.050400
300,0.048000
350,0.038800
400,0.037700
450,0.040400


***** Running evaluation *****


{'accuracy': 0.904}


In [90]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

train_texts = train_dataset["text"]
train_labels = train_dataset["label"]

train_embeddings = SF_model.model_body.encode(train_texts)
X_train = torch.tensor(train_embeddings, dtype=torch.float32).to(device)
y_train = torch.tensor(train_labels, dtype=torch.long).to(device)

eval_texts = eval_dataset["text"]
eval_labels = eval_dataset["label"]

eval_embeddings = SF_model.model_body.encode(eval_texts)
X_eval = torch.tensor(eval_embeddings, dtype=torch.float32).to(device)
y_eval = torch.tensor(eval_labels, dtype=torch.long).to(device)


In [93]:
def train_loop(X, y, model, loss_fn, optimizer):
    
    model.train()
    # Compute prediction and loss
    optimizer.zero_grad()
    pred = model(X)
    loss = loss_fn(pred, y) 
    # Backpropagation
    loss.backward()
    optimizer.step()
    return loss.item()

def test_loop(X, y, model, loss_fn):
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        pred = model(X)
        test_loss += loss_fn(pred, y).item()
        correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    accuracy = correct / len(y)
    print(f"Test Error: \n Accuracy: {(100*accuracy):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [110]:
labelsList = list(label_to_idx)

NN_model = IntentNN(len(labelsList)).to(device)
learning_rate = 0.001

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(NN_model.parameters(), lr=learning_rate)

epochs = 600
for t in range(epochs):
    loss = train_loop(X_train, y_train, NN_model, loss_fn, optimizer)
    if (t+1) % 10 == 0:
        print(f"Epoch {t+1} \n------------------------------ \nloss: {loss:.4f}")
        test_loop(X_eval, y_eval, NN_model, loss_fn)
print("Done!")

Using cuda device
Epoch 10 
------------------------------ 
loss: 4.3524
Test Error: 
 Accuracy: 41.2%, Avg loss: 4.342594 

Epoch 20 
------------------------------ 
loss: 4.1362
Test Error: 
 Accuracy: 56.4%, Avg loss: 4.121731 

Epoch 30 
------------------------------ 
loss: 3.7731
Test Error: 
 Accuracy: 62.0%, Avg loss: 3.754098 

Epoch 40 
------------------------------ 
loss: 3.2654
Test Error: 
 Accuracy: 63.2%, Avg loss: 3.239885 

Epoch 50 
------------------------------ 
loss: 2.6676
Test Error: 
 Accuracy: 68.4%, Avg loss: 2.652321 

Epoch 60 
------------------------------ 
loss: 2.0991
Test Error: 
 Accuracy: 72.4%, Avg loss: 2.106152 

Epoch 70 
------------------------------ 
loss: 1.6472
Test Error: 
 Accuracy: 76.8%, Avg loss: 1.669544 

Epoch 80 
------------------------------ 
loss: 1.3192
Test Error: 
 Accuracy: 82.4%, Avg loss: 1.350551 

Epoch 90 
------------------------------ 
loss: 1.0870
Test Error: 
 Accuracy: 84.8%, Avg loss: 1.124508 

Epoch 100 
--------

In [111]:
SF_model.model_body.save_pretrained(BASE_DIR / "models" / "embeddings" / "minilm")
print("SF model saved.")

SF model saved.


In [112]:
torch.save(NN_model.state_dict(), NN_MODEL_DIR / "intent_nn.pt")
print("NN model saved.")

NN model saved.
